# MSI5102 — Part II: Alternative Model Comparison
**Models:** ConvNeXt-T and ResNet50 with ImageNet pretrained weights + AutoAugment  
**Dataset:** PathMNIST (224x224, 9 classes)  
**Argument:** Reproducing the paper's baselines under fair training conditions to assess whether MedMamba's gains are architectural or a result of training condition disparities.

## STEP 1 — Mount Google Drive and Install Dependencies

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install scikit-learn --quiet

## STEP 2 — Imports and Configuration

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.models import (
    convnext_tiny, ConvNeXt_Tiny_Weights,
    resnet50, ResNet50_Weights
)
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize
from PIL import Image
import time, os

DRIVE_PATH = "/content/drive/MyDrive/5102 Project"
NPZ_PATH   = f"{DRIVE_PATH}/pathmnist_224.npz"
CKPT_DIR   = f"{DRIVE_PATH}/part2_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


## STEP 3 — Load PathMNIST from .npz


In [3]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

NPZ_PATH = '/content/drive/MyDrive/5102 Project/pathmnist_224.npz'

class NPZDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.fromarray(self.images[idx].astype(np.uint8))
        label = int(self.labels[idx].flatten()[0])
        if self.transform:
            img = self.transform(img)
        return img, label

data = np.load(NPZ_PATH)
print("Keys:", list(data.keys()))
print(f"Train: {data['train_images'].shape}")
print(f"Val:   {data['val_images'].shape}")
print(f"Test:  {data['test_images'].shape}")
print("Loaded successfully.")

Keys: ['train_images', 'train_labels', 'val_images', 'val_labels', 'test_images', 'test_labels']
Train: (89996, 224, 224, 3)
Val:   (10004, 224, 224, 3)
Test:  (7180, 224, 224, 3)
Loaded successfully.


## STEP 4 — Define Transforms

**Key difference from the paper:** We use ImageNet normalisation (matching pretrained weight expectations) and AutoAugment for training. The paper used no augmentation and no pretraining for its baselines. This is the core of our methodological critique.

In [6]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_dataset = NPZDataset(data['train_images'], data['train_labels'], train_transform)
val_dataset   = NPZDataset(data['val_images'],   data['val_labels'],   eval_transform)
test_dataset  = NPZDataset(data['test_images'],  data['test_labels'],  eval_transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=128, shuffle=False,
                          num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=128, shuffle=False,
                          num_workers=0, pin_memory=True)

print(f"Train: {len(train_dataset):,} | Val: {len(val_dataset):,} | Test: {len(test_dataset):,}")

Train: 89,996 | Val: 10,004 | Test: 7,180


## STEP 5 — Training and Evaluation Functions

In [10]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, num_classes=9):
    model.eval()
    all_labels, all_probs = [], []
    correct, total = 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs = model(imgs)
        probs   = torch.softmax(outputs, dim=1)
        correct += (outputs.argmax(1) == labels).sum().item()
        total   += imgs.size(0)
        all_labels.append(labels.cpu().numpy())
        all_probs.append(probs.cpu().numpy())

    all_labels = np.concatenate(all_labels)
    all_probs  = np.concatenate(all_probs)
    oa         = correct / total

    labels_bin = label_binarize(all_labels, classes=list(range(num_classes)))
    auc = roc_auc_score(labels_bin, all_probs,
                        multi_class='ovr', average='macro')
    return oa, auc


def run_training(model, model_name, epochs=10, lr=0.001):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr,
                            betas=(0.9, 0.999), weight_decay=1e-4)
    scheduler = optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=[7, 9], gamma=0.1
    )

    best_val_oa = 0.0
    best_ckpt   = f"{CKPT_DIR}/{model_name}_best.pth"

    print(f"\n{'='*60}")
    print(f"  Training {model_name}  |  {epochs} epochs  |  lr={lr}")
    print(f"{'='*60}")
    print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Train OA':>8} | {'Val OA':>8} | {'Val AUC':>8} | {'Time':>6}")
    print("-"*60)

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss, train_oa = train_one_epoch(model, train_loader, optimizer, criterion)
        val_oa, val_auc      = evaluate(model, val_loader)
        scheduler.step()

        if val_oa > best_val_oa:
            best_val_oa = val_oa
            torch.save(model.state_dict(), best_ckpt)

        # Print every epoch since we only have 10
        print(f"{epoch:>6} | {train_loss:>10.4f} | {train_oa:>8.4f} | "
              f"{val_oa:>8.4f} | {val_auc:>8.4f} | {time.time()-t0:>5.1f}s")

    print(f"\nBest Val OA: {best_val_oa:.4f}")
    print(f"Loading best checkpoint...")
    model.load_state_dict(torch.load(best_ckpt))
    test_oa, test_auc = evaluate(model, test_loader)

    print(f"\n{'*'*60}")
    print(f"  {model_name} — FINAL TEST RESULTS")
    print(f"  OA  = {test_oa:.4f}  ({test_oa*100:.2f}%)")
    print(f"  AUC = {test_auc:.4f}")
    print(f"{'*'*60}")

    return test_oa, test_auc

print("Functions defined.")

Functions defined.


## STEP 6 — Train ConvNeXt-T

ConvNeXt-T is the paper's **own chosen baseline**, trained here with ImageNet pretrained weights. This directly exposes the unfair comparison in the original paper.

In [11]:
convnext_model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
convnext_model.classifier[2] = nn.Linear(
    convnext_model.classifier[2].in_features, 9
)

print(f"ConvNeXt-T output classes: {convnext_model.classifier[2].out_features}")
print(f"Parameters: {sum(p.numel() for p in convnext_model.parameters()):,}")

convnext_oa, convnext_auc = run_training(
    convnext_model, "ConvNeXt-T", epochs=10, lr=0.001
)

ConvNeXt-T output classes: 9
Parameters: 27,827,049

  Training ConvNeXt-T  |  10 epochs  |  lr=0.001
 Epoch | Train Loss | Train OA |   Val OA |  Val AUC |   Time
------------------------------------------------------------
     1 |     0.2196 |   0.9271 |   0.9089 |   0.9989 | 1919.2s
     2 |     0.1218 |   0.9608 |   0.9825 |   0.9997 | 1908.2s
     3 |     0.0977 |   0.9678 |   0.9795 |   0.9996 | 1902.4s
     4 |     0.0911 |   0.9710 |   0.9451 |   0.9994 | 1903.5s
     5 |     0.0852 |   0.9718 |   0.9842 |   0.9998 | 1901.9s
     6 |     0.0777 |   0.9749 |   0.9862 |   0.9997 | 1900.5s
     7 |     0.0693 |   0.9772 |   0.9865 |   0.9998 | 1902.6s
     8 |     0.0271 |   0.9909 |   0.9963 |   1.0000 | 1900.6s
     9 |     0.0176 |   0.9942 |   0.9957 |   1.0000 | 1902.4s
    10 |     0.0148 |   0.9950 |   0.9957 |   1.0000 | 1902.8s

Best Val OA: 0.9963
Loading best checkpoint...

************************************************************
  ConvNeXt-T — FINAL TEST RESULTS
 

## STEP 7 — Train ResNet50

ResNet50 serves as a second data point. If both ConvNeXt-T and ResNet50 close the gap under fair conditions, the finding is not architecture-specific — it is a training conditions problem.

In [12]:
resnet_model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, 9)

print(f"ResNet50 output classes: {resnet_model.fc.out_features}")
print(f"Parameters: {sum(p.numel() for p in resnet_model.parameters()):,}")

resnet_oa, resnet_auc = run_training(
    resnet_model, "ResNet50", epochs=10, lr=0.001
)

ResNet50 output classes: 9
Parameters: 23,526,473

  Training ResNet50  |  10 epochs  |  lr=0.001
 Epoch | Train Loss | Train OA |   Val OA |  Val AUC |   Time
------------------------------------------------------------
     1 |     0.3079 |   0.8979 |   0.9388 |   0.9981 | 1227.2s
     2 |     0.1681 |   0.9445 |   0.9710 |   0.9993 | 1221.3s
     3 |     0.1384 |   0.9544 |   0.9789 |   0.9996 | 1221.6s
     4 |     0.1190 |   0.9607 |   0.9696 |   0.9993 | 1228.0s
     5 |     0.1115 |   0.9637 |   0.9746 |   0.9994 | 1225.0s
     6 |     0.1008 |   0.9670 |   0.9796 |   0.9995 | 1223.5s
     7 |     0.0969 |   0.9685 |   0.9703 |   0.9993 | 1221.1s
     8 |     0.0501 |   0.9839 |   0.9941 |   0.9999 | 1214.7s
     9 |     0.0397 |   0.9869 |   0.9937 |   0.9999 | 1216.6s
    10 |     0.0346 |   0.9889 |   0.9948 |   0.9999 | 1215.6s

Best Val OA: 0.9948
Loading best checkpoint...

************************************************************
  ResNet50 — FINAL TEST RESULTS
  OA  =

## STEP 8 — Final Results Summary

In [13]:
print("\n" + "="*65)
print("  PART II — FINAL RESULTS SUMMARY (PathMNIST, 9 classes)")
print("="*65)
print(f"{'Model':<38} {'OA (%)':>7} {'AUC':>8}  {'Conditions'}")
print("-"*65)
print(f"{'MedMamba-T  (paper)':<38} {'95.30':>7} {'0.9970':>8}  No pretrain, no aug")
print(f"{'MedMamba-S  (paper)':<38} {'95.50':>7} {'0.9970':>8}  No pretrain, no aug")
print(f"{'ResNet50    (paper)':<38} {'89.20':>7} {'0.9890':>8}  No pretrain, no aug")
print("-"*65)
print(f"{'ConvNeXt-T  (ours)':<38} {convnext_oa*100:>7.2f} {convnext_auc:>8.4f}  ImageNet + AutoAugment")
print(f"{'ResNet50    (ours)':<38} {resnet_oa*100:>7.2f} {resnet_auc:>8.4f}  ImageNet + AutoAugment")
print("="*65)


  PART II — FINAL RESULTS SUMMARY (PathMNIST, 9 classes)
Model                                   OA (%)      AUC  Conditions
-----------------------------------------------------------------
MedMamba-T  (paper)                      95.30   0.9970  No pretrain, no aug
MedMamba-S  (paper)                      95.50   0.9970  No pretrain, no aug
ResNet50    (paper)                      89.20   0.9890  No pretrain, no aug
-----------------------------------------------------------------
ConvNeXt-T  (ours)                       97.59   0.9978  ImageNet + AutoAugment
ResNet50    (ours)                       96.28   0.9981  ImageNet + AutoAugment
